In [172]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

In [173]:
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

In [174]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

num_test_samples = mnist_info.splits['test'].num_examples

num_test_samples = tf.cast(num_test_samples, tf.int64)


def scale(image, label):
    
    image = tf.cast(image, tf.float32)
    
    image /= 255.

    return image, label


scaled_train_and_validation_data = mnist_train.map(scale)

test_data = mnist_test.map(scale)


BUFFER_SIZE = 10000

shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

validation_data = shuffled_train_and_validation_data.take(num_validation_samples)

train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

BATCH_SIZE = 500

train_data = train_data.batch(BATCH_SIZE)

validation_data = validation_data.batch(num_validation_samples)

test_data = test_data.batch(num_test_samples)

validation_inputs, validation_targets = next(iter(validation_data))

In [175]:
input_size = 784
output_size = 10
hidden_layer_size = 400

model = tf.keras.Sequential([
                             tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
                             tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
                             tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
                             tf.keras.layers.Dense(output_size, activation='softmax')
                            ])

In [180]:
custom_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=custom_optimizer , loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [181]:
NUM_EPOCHS = 50

model.fit(train_data, epochs=NUM_EPOCHS, validation_data=(validation_inputs, validation_targets), verbose=2)

Epoch 1/5
108/108 - 2s - 18ms/step - accuracy: 0.9772 - loss: 0.0717 - val_accuracy: 0.9745 - val_loss: 0.0950
Epoch 2/5
108/108 - 1s - 11ms/step - accuracy: 0.9811 - loss: 0.0602 - val_accuracy: 0.9755 - val_loss: 0.0886
Epoch 3/5
108/108 - 1s - 12ms/step - accuracy: 0.9828 - loss: 0.0559 - val_accuracy: 0.9755 - val_loss: 0.0845
Epoch 4/5
108/108 - 1s - 12ms/step - accuracy: 0.9838 - loss: 0.0526 - val_accuracy: 0.9760 - val_loss: 0.0820
Epoch 5/5
108/108 - 1s - 12ms/step - accuracy: 0.9843 - loss: 0.0509 - val_accuracy: 0.9767 - val_loss: 0.0800


In [182]:
test_loss, test_accuracy = model.evaluate(test_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 701ms/step - accuracy: 0.9705 - loss: 0.1091


In [183]:
print('Test loss {0:.2f}. Test accuracy: {1:.2f}%' .format(test_loss, test_accuracy*100.))

Test loss 0.11. Test accuracy: 97.05%
